In [0]:
-- Check if silver tables were created
SHOW TABLES IN adtech_catalog.silver;

database,tableName,isTemporary
silver,conformed_ad_catalog,false
silver,conformed_user_clicks,false


In [0]:
-- Count rows in conformed_user_clicks
-- Expected: ~500,000 rows
SELECT 
    'conformed_user_clicks' as table_name,
    COUNT(*) as row_count
FROM adtech_catalog.silver.conformed_user_clicks;

table_name,row_count
conformed_user_clicks,2500000


In [0]:
-- Count rows in conformed_ad_catalog
-- Expected: ~1,000 rows
SELECT 
    'conformed_ad_catalog' as table_name,
    COUNT(*) as row_count
FROM adtech_catalog.silver.conformed_ad_catalog;

table_name,row_count
conformed_ad_catalog,9999


In [0]:
-- Summary of both tables
SELECT 
    'conformed_user_clicks' as table_name,
    COUNT(*) as row_count
FROM adtech_catalog.silver.conformed_user_clicks
UNION ALL
SELECT 
    'conformed_ad_catalog' as table_name,
    COUNT(*) as row_count
FROM adtech_catalog.silver.conformed_ad_catalog;

table_name,row_count
conformed_user_clicks,2500000
conformed_ad_catalog,9999


In [0]:
-- Sample from conformed_user_clicks (check cleaned data)
SELECT * 
FROM adtech_catalog.silver.conformed_user_clicks 
LIMIT 5;

-- Sample from conformed_ad_catalog
SELECT * 
FROM adtech_catalog.silver.conformed_ad_catalog 
LIMIT 5;

Ad_Reference_ID,Ad_Category_Standard,Ad_Device_Cleaned,Ad_Location_Cleaned,Cost_Per_Click_Cleaned,Ad_Type_Standard,ad_video_length_cleaned,ingestion_timestamp,ingestion_date,source_file,ingestion_batch_id,processing_timestamp,processing_batch_id,processing_status,environment
AD_100146,Food,Mobile,Delhi,1.14,Video,60.0,2026-08-02T10:11:37.099Z,2026-08-02,ad_catalog_raw.csv,20260802_101104,2026-08-02T10:32:11.711Z,20260802_103154,CLEANED,development
AD_100248,Health,Mobile,Delhi,2.83,Unknown,0.0,2026-08-02T10:11:37.099Z,2026-08-02,ad_catalog_raw.csv,20260802_101104,2026-08-02T10:32:11.711Z,20260802_103154,CLEANED,development
AD_100322,Electronics,Tablet,Karnataka,3.00,Text,0.0,2026-08-02T10:11:37.099Z,2026-08-02,ad_catalog_raw.csv,20260802_101104,2026-08-02T10:32:11.711Z,20260802_103154,CLEANED,development
AD_100380,Food,All-Devices,Maharashtra,0.70,Video,15.0,2026-08-02T10:11:37.099Z,2026-08-02,ad_catalog_raw.csv,20260802_101104,2026-08-02T10:32:11.711Z,20260802_103154,CLEANED,development
AD_100469,Electronics,All-Devices,Karnataka,2.28,Video,15.0,2026-08-02T10:11:37.099Z,2026-08-02,ad_catalog_raw.csv,20260802_101104,2026-08-02T10:32:11.711Z,20260802_103154,CLEANED,development


In [0]:
-- Check for remaining anomalies after cleaning
SELECT 
    COUNT(*) as total_rows,
    SUM(CASE WHEN user_age_cleaned < 18 THEN 1 ELSE 0 END) as underage_users,
    SUM(CASE WHEN user_age_cleaned > 100 THEN 1 ELSE 0 END) as overage_users,
    SUM(CASE WHEN Watch_Duration_Cleaned < 0 THEN 1 ELSE 0 END) as negative_duration,
    SUM(CASE WHEN Watch_Duration_Cleaned > 180 THEN 1 ELSE 0 END) as overflow_duration,
    SUM(CASE WHEN device_cleaned = 'Unknown' THEN 1 ELSE 0 END) as unknown_device,
    SUM(CASE WHEN user_clicked_cleaned = 1 AND Watch_Duration_Cleaned = 0 THEN 1 ELSE 0 END) as logical_conflicts
FROM adtech_catalog.silver.conformed_user_clicks;

total_rows,underage_users,overage_users,negative_duration,overflow_duration,unknown_device,logical_conflicts
2500000,0,0,0,0,416739,0


In [0]:
-- Check for remaining anomalies in ad catalog
SELECT 
    COUNT(*) as total_rows,
    SUM(CASE WHEN Ad_Category_Standard = 'Unknown' THEN 1 ELSE 0 END) as unknown_category,
    SUM(CASE WHEN Cost_Per_Click_Cleaned < 0 THEN 1 ELSE 0 END) as negative_cpc,
    SUM(CASE WHEN Cost_Per_Click_Cleaned IS NULL THEN 1 ELSE 0 END) as null_cpc,
    SUM(CASE WHEN ad_video_length_cleaned < 0 THEN 1 ELSE 0 END) as negative_video_length,
    SUM(CASE WHEN ad_video_length_cleaned > 60 THEN 1 ELSE 0 END) as long_video
FROM adtech_catalog.silver.conformed_ad_catalog;

total_rows,unknown_category,negative_cpc,null_cpc,negative_video_length,long_video
9999,0,0,0,0,0


In [0]:
-- See how ads are distributed by category
SELECT 
    Ad_Category_Standard,
    COUNT(*) as ad_count
FROM adtech_catalog.silver.conformed_ad_catalog
GROUP BY Ad_Category_Standard
ORDER BY ad_count DESC;

Ad_Category_Standard,ad_count
Electronics,1904
Fashion,1658
Health,1651
Food,1609
Gaming,1606
Travel,1571


In [0]:
-- See how users are distributed by device
SELECT 
    device_cleaned,
    COUNT(*) as user_count
FROM adtech_catalog.silver.conformed_user_clicks
GROUP BY device_cleaned
ORDER BY user_count DESC;

device_cleaned,user_count
Desktop,833288
Mobile,833124
Tablet,416849
Unknown,416739


In [0]:
-- Check if Silver run was logged
SELECT 
    version_id,
    deployed_at,
    description,
    status
FROM adtech_catalog.monitoring.version_history
WHERE description LIKE '%Silver%'
ORDER BY deployed_at DESC;

version_id,deployed_at,description,status
20260802_103154,2026-08-02T10:32:40.231671,Silver Layer - Data Cleaning,SUCCESS


In [0]:
-- Check all pipeline runs
SELECT 
    description,
    COUNT(*) as total_runs,
    SUM(CASE WHEN status = 'SUCCESS' THEN 1 ELSE 0 END) as successful,
    SUM(CASE WHEN status != 'SUCCESS' THEN 1 ELSE 0 END) as failed
FROM adtech_catalog.monitoring.version_history
GROUP BY description
ORDER BY total_runs DESC;

description,total_runs,successful,failed
Gold Layer - Feature Engineering,1,1,0
Anomaly Detection - Bronze Layer,1,1,0
Silver Layer - Data Cleaning,1,1,0
Bronze Layer - Initial Load,1,1,0
Gold Quality Review,1,1,0
Export Gold to S3,1,1,0


In [0]:
-- One-line summary of Silver layer
SELECT 
    'SILVER LAYER' as layer,
    COUNT(*) as total_rows,
    'PASSED' as quality_status
FROM adtech_catalog.silver.conformed_user_clicks
UNION ALL
SELECT 
    'SILVER CATALOG' as layer,
    COUNT(*) as total_rows,
    'PASSED' as quality_status
FROM adtech_catalog.silver.conformed_ad_catalog;

layer,total_rows,quality_status
SILVER LAYER,2500000,PASSED
SILVER CATALOG,9999,PASSED
